# S4 NLL: build the figure data

Turns the raw inputs into **one pickle**, `s4_figure_data.pkl`, that holds every table
`s4_nll_figures.ipynb` needs for Fig. 6 C-I and Supplementary Fig. S8 A-E. The figure
notebook reads that file and nothing else.

Like `data_figs.ipynb`, this notebook runs only on the Gentner lab cluster. It starts from the saved
output of the S4 models (one model per bird, trained on its regular air song; Methods 4.4.2), the wavs
they scored, the two export pickles of the S4 analysis notebooks, the syllable tables, and three sets of
files that `data_figs.ipynb` writes, so `data_figs.ipynb` runs first. The output saved under each cell is
from the run that made `data/s4_figure_data.pkl`.

| Table in the pickle | Rows are | Built from | Used for |
|---|---|---|---|
| `region_nll` | song x region (silence, song) | export pickles | Fig. 6C, S8 A-B |
| `boundary_nll` | syllable x region (boundary, interior) | export pickles | Fig. 6 E-F |
| `syllable_position_nll` | syllable x position bin, with the bird | export pickles | Fig. 6D |
| `song_covariates` | song, with amplitude, entropy, duration | export pickles + amplitude file + BioSound files | Fig. 6I |
| `starling_syllable_nll` | starling syllable | wavs + S4 output + syllable tables | Fig. 6H, S8E |
| `starling_kgram_sets` | (bird, condition, k) -> retained k-grams | k-gram notebook export | Fig. 6H, S8E |
| `zf_syllable_nll` | zebra finch syllable | wavs + S4 output + syllable table | Fig. 6G |
| `song_trace_metrics` | song, with peak NLL, jump rate, amplitude | wavs + S4 output + amplitude file | S8 C-D |

**Amplitude.** The amplitude of a song is the mean over its syllables of the RMS amplitude of
the recorded waveform, in dB re full scale. It is measured in `data_figs.ipynb` (Cell A1), which
writes `syllable_level_data.pkl`; Cell L1 reads that file and attaches one value to every song.
The columns are called `mean_level_db` (in `song_covariates`) and `level_db` (in
`song_trace_metrics`), the names they have in that file. Amplitude replaces the two earlier
"power" columns, which did not measure how loud a song was and are no longer saved.

**Entropy.** The entropy of Fig. 6I is BioSound's spectral entropy (`entS`), averaged over the syllables
of each song and saved as `mean_ent_s`. Cell B1 takes it from the two small BioSound files per bird
that `data_figs.ipynb` keeps (`biosound_paf_slim/` from Cell A4 and `biosound_slim_day/` from Cell E3).
The syllable tables' own `spectral_entropy`, which the panel used before, stays in the file as
`mean_spectral_entropy`.

**Two ways to run.** The three audio-derived tables are slow to make, so each is kept in
`s4_figure_cache/`. With `REBUILD_FROM_AUDIO = False` (the default) the notebook reuses
those files when all three exist and finishes in seconds. Set it to `True` to recompute
them from the wavs and model outputs. A rebuild is compared against the existing cache
before anything is overwritten, so a change in the numbers cannot go unnoticed.

**Run order:** top to bottom. Section 2 (Cells P1-P3) is skipped unless `REPAIR_EXPORTS = True`
in Cell 1. Cells L1 and B1 have to run before Cell 13, in that order, and Cell 13 stops with a message if either has not.

## 1. Configuration

In [1]:
# Cell 1: paths and run control
import os


def _path(env_name, default):
    # Each location can be overridden from the environment, so the notebook runs unchanged
    # on another machine: export HELIOX_S4_DIR=/somewhere/else before starting Jupyter.
    return os.environ.get(env_name, default)


PROC_DIR = _path('HELIOX_PROC_DIR',
                 '/mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing')
S4_DIR = _path('HELIOX_S4_DIR', '/mnt/cube/finch/marissa/s4_analysis')

DATA_ROOT = os.path.join(S4_DIR, 'data')        # one folder per bird, wavs under processed/
MODELS_ROOT = os.path.join(S4_DIR, 'models')    # *s4_genpred_dict_{regair,heliox}_full.pkl
BASE_DIR = os.path.join(PROC_DIR, 'data')
DERIVED = os.path.join(BASE_DIR, 'derived')
STARLING_SYL_DIR = os.path.join(PROC_DIR, 'figures', 'starling_runs')

STARLING_EXPORT_PATH = os.path.join(BASE_DIR, 'starling_combined_export.pkl')
ZF_EXPORT_PATH = os.path.join(BASE_DIR, 'zf_combined', 'zf_combined_export.pkl')
ZF_SYL_PATH = os.path.join(BASE_DIR, 'zf_combined', 'syldf_fullycombined_20250807.pickle')
MOTIF_PKL = os.path.join(DERIVED, 'starling_final_motif_sets.pkl')

CACHE_DIR = os.path.join(DERIVED, 's4_figure_cache')
BUNDLE_PATH = _path('HELIOX_S4_BUNDLE', os.path.join(DERIVED, 's4_figure_data.pkl'))
os.makedirs(CACHE_DIR, exist_ok=True)

# False: reuse the three cached tables if they all exist. True: recompute them from audio.
REBUILD_FROM_AUDIO = False
# False: skip Cells P1-P3. True: rebuild the export tables behind Fig. 6D and 6I (Section 2).
# The exports on disk already hold that repair, so this stays False in a normal run.
REPAIR_EXPORTS = False

CACHE_FILES = {
    'starling_syllable_nll': os.path.join(CACHE_DIR, 'starling_sn.pkl'),
    'zf_syllable_nll': os.path.join(CACHE_DIR, 'zf_full_syl_df.pkl'),
    'song_trace_metrics': os.path.join(CACHE_DIR, 'adj_df.pkl'),
}
MANIFEST_PATH = os.path.join(CACHE_DIR, 's4_row_manifest.csv')

FROM_AUDIO = REBUILD_FROM_AUDIO or not all(os.path.exists(p) for p in CACHE_FILES.values())
print('bundle ->', BUNDLE_PATH)
print('mode   :', 'recompute from audio' if FROM_AUDIO else 'reuse cached tables')
print('exports:', 'rebuild the Fig. 6D and 6I tables (Cells P1-P3)' if REPAIR_EXPORTS
      else 'used as they are (Cells P1-P3 skipped)')

bundle -> /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/s4_figure_data.pkl
mode   : reuse cached tables
exports: used as they are (Cells P1-P3 skipped)


In [2]:
# Cell 2: analysis parameters
# Rationale: every threshold the tables depend on lives here, and the whole dict is written
# into the pickle, so the figure notebook and any reader can see how the data were made.
SAMPLE_RATE = 24000            # the NLL trace has one entry per audio sample, both species
BIN_MS = 5.0                   # traces are mean-pooled into 5 ms bins before any metric
MIN_SONG_S = {'starling': 10.0, 'zebra_finch': 0.0}   # zf motifs run far shorter than bouts
ZF_EDGE_MS = 10                # trimmed from each end of a zf syllable (onset/offset transients)
MIN_BINS_PER_WINDOW = 2        # a starling syllable or gap needs this many bins to get a mean
JUMP_THRESH_STD = 2.0          # jump = |frame-to-frame change| > this many robust SDs
ENV_SMOOTH_MS = 5.0            # Hann smoothing of the Hilbert envelope
SILENCE_FRAC_OF_PEAK = 0.02    # song = envelope >= 2% of the bout maximum (matches Methods)

LABCOL = 'hdbscan_labels_new_0_005'   # 0.5% min_cluster_size, the clustering in the paper
BOUTCOL = 'bout_index'
POSCOL = 'syllables_sequence_pos'

# starling names as they appear in the syllable tables (and in the k-gram sets)
STARLING_BIRDS = ['s_b1556h_24', 's_b1484h_24', 's_b1364_25',
                  's_b1354_25', 's_b1481h_24', 's_b1349h_23']

# One starling's model files carry no prefix at all; they belong to this bird number.
UNPREFIXED_MODEL_BIRD = '1364'

# Every table must hold this many birds in BOTH conditions. A bird whose model file or wav
# folder fails to resolve would otherwise drop out silently and the models would still fit.
EXPECTED_BIRDS = {'starling': 6, 'zebra_finch': 6}
# False: stop when a table is short of birds. True: say so loudly, record it in the
# pickle, and carry on. Leave it False unless a shortfall is understood and intended.
ALLOW_INCOMPLETE = False

# Left out of the k-gram NLL analysis on purpose. This bird's k-gram sets were selected on
# the s_b1364h_25 syllable table, whose cluster numbering differs from the labels in the
# NLL table (s_b1364_25): 0-5% of its selected sequences occur there, the same as with
# scrambled labels. Applying them would give meaningless inside/outside labels.
KGRAM_NLL_EXCLUDE = {'s_b1364_25': 'k-gram sets come from a different clustering (s_b1364h_25)'}

COND_ORDER = ['regair', 'heliox']
BIN_SAMPLES = max(1, int(round(BIN_MS / 1000.0 * SAMPLE_RATE)))

PARAMS = dict(SAMPLE_RATE=SAMPLE_RATE, BIN_MS=BIN_MS, MIN_SONG_S=MIN_SONG_S, ZF_EDGE_MS=ZF_EDGE_MS,
              MIN_BINS_PER_WINDOW=MIN_BINS_PER_WINDOW, JUMP_THRESH_STD=JUMP_THRESH_STD,
              ENV_SMOOTH_MS=ENV_SMOOTH_MS, SILENCE_FRAC_OF_PEAK=SILENCE_FRAC_OF_PEAK,
              LABCOL=LABCOL, EXPECTED_BIRDS=EXPECTED_BIRDS, KGRAM_NLL_EXCLUDE=KGRAM_NLL_EXCLUDE)

In [3]:
# Cell 3: imports and shared helpers
import re, glob, pickle, shutil, datetime
import numpy as np
import pandas as pd
from scipy.io.wavfile import read as wavread
from scipy.signal import hilbert


class _NumpyCompatUnpickler(pickle.Unpickler):
    """Redirects numpy._core.* <-> numpy.core.* during unpickling, so results saved under
    one numpy major version load under the other."""
    def find_class(self, module, name):
        if module.startswith('numpy._core'):
            module = module.replace('numpy._core', 'numpy.core', 1)
        elif module.startswith('numpy.core'):
            alt = module.replace('numpy.core', 'numpy._core', 1)
            try:
                return super().find_class(alt, name)
            except ModuleNotFoundError:
                pass
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, 'rb') as f:
        try:
            return pickle.load(f)
        except ModuleNotFoundError as e:
            if '_core' not in str(e) and 'numpy' not in str(e):
                raise
            f.seek(0)
            return _NumpyCompatUnpickler(f).load()


def bird_num(s):
    """'s_b1556h_24' -> '1556'; also works on S4 folder names."""
    m = re.search(r'b(\d+)', str(s))
    return m.group(1) if m else None


def read_wav_mono_float(path):
    sr, x = wavread(path)
    if x.ndim > 1:
        x = x.mean(axis=1)
    x = x.astype(np.float64)
    x /= (np.max(np.abs(x)) + 1e-12)
    return sr, x


def bin_trace(x, bin_samples):
    """Mean-pool a sample-rate trace into non-overlapping bins."""
    n_bins = len(x) // bin_samples
    if n_bins < 3:
        return np.array([])
    return x[:n_bins * bin_samples].reshape(n_bins, bin_samples).mean(axis=1)


INCOMPLETE = []     # filled only when ALLOW_INCOMPLETE lets a short table through


def require_complete(df, name, bird_col, cond_col, species=None, species_col=None):
    """Stop if any species x condition cell is short of birds. `species` names the species
    of a single-species table; `species_col` is the column in a two-species one."""
    problems = []
    for sp, n_expected in EXPECTED_BIRDS.items():
        if species is not None and sp != species:
            continue
        sub = df if species_col is None else df[df[species_col] == sp]
        for cond in COND_ORDER:
            n = sub.loc[sub[cond_col] == cond, bird_col].nunique()
            if n != n_expected:
                problems.append(f'{sp} {cond}: {n} birds, expected {n_expected}')
    if not problems:
        print(f'{name}: complete ({len(df)} rows)')
        return
    message = f'{name} is missing birds -> ' + '; '.join(problems)
    if not ALLOW_INCOMPLETE:
        raise RuntimeError(message + '. Set ALLOW_INCOMPLETE = True in Cell 2 to continue anyway.')
    INCOMPLETE.append(message)
    print('[INCOMPLETE]', message)


print('helpers ready')

helpers ready


## 2. Repair the export pickles (run once)

The export pickles written by the S4 analysis notebooks had two gaps. Three zebra finches are
named with a `z_` in the syllable table, so they matched no syllables and were missing from the
entropy table (Fig. 6I) and the syllable-position table (Fig. 6D). The syllable-position
tables also had no bird column, so Fig. 6D could not weight birds equally.

Cells P1-P3 rebuild those tables for every bird from the wavs, the S4 output and the syllable
tables, check that the rebuild gives back what the exports already held, and only then save the
exports. The earlier file is kept beside the new one with a date in its name.

**This has been done**, so the cells are skipped unless `REPAIR_EXPORTS = True` in Cell 1. Turn it
on in two cases: the S4 analysis notebooks have written the exports again, or `s4_song_key.csv`
has been deleted from `s4_figure_cache/` (Cell L1 takes the song names from P1 and P2 the first
time it runs). If the exports already hold the repair, P3 says so and writes nothing.

In [4]:
# Cell P1: rebuild the zebra finch tables behind Fig. 6D and 6I, for all six birds
# The S4 analysis notebook looked each bird up in the syllable table as its folder name minus 'zf_'
# ('c8c19_24'). Three birds are named with a 'z_' there ('z_c8c19_24'), so they matched no syllables
# and dropped out of the power / entropy table and the syllable-position table. Everything below
# follows that notebook step for step; the only change is the lookup, which now tries both names.
if REPAIR_EXPORTS:
    N_POSITION_BINS = 20     # position bins per syllable, as in the S4 analysis notebooks


    def wavs_as_scored(folder):
        """Row i of a model's output is the i-th wav in plain os.listdir order (see Cell 6), so the
        order is left exactly as the filesystem gives it."""
        return [f for f in os.listdir(folder) if f.lower().endswith('.wav')]


    def scored_songs(bird_id, condition, wav_dir, model_path, min_song_s):
        """One row per scored song (file, mean NLL over the unpadded part, duration) and the traces."""
        files = wavs_as_scored(wav_dir)
        traces = load_pickle(model_path)['logprobs_array_full']
        if len(files) != traces.shape[0]:
            raise RuntimeError(f'{bird_id}/{condition}: {len(files)} wavs but {traces.shape[0]} model rows')
        lengths = np.array([len(wavread(os.path.join(wav_dir, f))[1]) for f in files])
        keep = lengths >= min_song_s * SAMPLE_RATE
        traces, lengths, files = traces[keep], lengths[keep], [f for f, k in zip(files, keep) if k]
        avg = np.full(len(files), np.nan)
        for i in range(len(files)):
            avg[i] = traces[i, :min(lengths[i], traces.shape[1])].mean()
        table = pd.DataFrame({'bird_id': bird_id, 'condition': condition, 'filename': files,
                              'song_idx': np.arange(len(files)), 's4_avg_nll': avg,
                              'duration_s': lengths / SAMPLE_RATE})
        return table, traces


    def position_profile(trace, start_s, end_s):
        """A syllable's NLL in N_POSITION_BINS equal slices; None if it is shorter than that many
        samples or runs past the trace."""
        on, off = int(start_s * SAMPLE_RATE), int(end_s * SAMPLE_RATE)
        if off <= on or off > trace.shape[0]:
            return None
        seg = trace[on:off]
        if len(seg) < N_POSITION_BINS:
            return None
        trim = (len(seg) // N_POSITION_BINS) * N_POSITION_BINS
        return seg[:trim].reshape(N_POSITION_BINS, -1).mean(axis=1)


    def position_table(profiles):
        """(bird_id, condition, profile) for each syllable -> one row per syllable x bin."""
        return pd.DataFrame({'bird_id': np.repeat([p[0] for p in profiles], N_POSITION_BINS),
                             'condition': np.repeat([p[1] for p in profiles], N_POSITION_BINS),
                             'bin': np.tile(np.arange(N_POSITION_BINS), len(profiles)),
                             'nll': np.concatenate([p[2] for p in profiles])})


    zf_syl = pd.read_pickle(ZF_SYL_PATH).reset_index(drop=True)
    zf_syl['heliox'] = zf_syl['heliox'].astype(bool)
    zf_names = set(zf_syl['indv'].astype(str).unique())


    def zf_name_in_table(folder):
        """'zf_c8c19_24' -> that bird's name in the syllable table, with or without 'z_'."""
        candidate = folder.replace('zf_', '', 1)
        for name in (candidate, 'z_' + candidate):
            if name in zf_names:
                return name
        raise KeyError(f'{folder}: no bird named {candidate} or z_{candidate} in the zebra finch syllable table')


    zf_song_rows = zf_syl.groupby([zf_syl['indv'].astype(str), 'heliox', zf_syl['song_id'].astype(str)],
                                  sort=False).groups
    model_files = sorted(glob.glob(os.path.join(MODELS_ROOT, '*.pkl')))


    def zf_model_file(folder, condition):
        suffix = f's4_genpred_dict_{condition}_full.pkl'
        for target in (folder + suffix, folder + '_' + suffix):
            for p in model_files:
                if os.path.basename(p) == target:
                    return p
        return None


    zf_tables, zf_position = [], []
    for d in sorted(x for x in glob.glob(os.path.join(DATA_ROOT, 'zf_*')) if os.path.isdir(x)):
        folder = os.path.basename(d.rstrip('/'))
        paths = {c: zf_model_file(folder, c) for c in COND_ORDER}
        if None in paths.values():
            continue                                    # not scored in both conditions
        indv = zf_name_in_table(folder)
        for cond, sub in [('regair', 'test'), ('heliox', 'song_heliox')]:
            table, traces = scored_songs(folder, cond, os.path.join(d, 'processed', sub), paths[cond],
                                         MIN_SONG_S['zebra_finch'])
            table['song_id'] = [os.path.splitext(f)[0] for f in table['filename']]
            table['indv'] = indv
            zf_tables.append(table)
            for song in table.itertuples():
                rows = zf_song_rows.get((indv, cond == 'heliox', song.song_id))
                if rows is None:
                    continue
                for syl in zf_syl.loc[rows, ['start_time', 'end_time']].itertuples():
                    profile = position_profile(traces[song.song_idx], syl.start_time, syl.end_time)
                    if profile is not None:
                        zf_position.append((folder, cond, profile))
            del traces
    zf_songs = pd.concat(zf_tables, ignore_index=True)
    zf_position = position_table(zf_position)

    # song-mean power and entropy of the syllables, matched to the scored songs on the corrected name
    zf_song_features = (zf_syl.groupby(['indv', 'heliox', 'song_id'])
                        .agg(mean_power=('avg_power', 'mean'), mean_spectral_entropy=('spectral_entropy', 'mean'))
                        .reset_index())
    zf_song_features['indv'] = zf_song_features['indv'].astype(str)
    zf_song_features['condition'] = np.where(zf_song_features['heliox'], 'heliox', 'regair')
    zf_song_features['song_id'] = zf_song_features['song_id'].astype(str)
    zf_confound = zf_songs.merge(zf_song_features, on=['indv', 'condition', 'song_id'], how='inner')

    print('zebra finch, per bird:')
    print(pd.DataFrame({'scored songs': zf_songs.groupby('bird_id').size(),
                        'with power / entropy': zf_confound.groupby('bird_id').size(),
                        'syllables in Fig. 6D': zf_position.groupby('bird_id').size() // N_POSITION_BINS})
          .fillna(0).astype(int).to_string())
    print('songs whose power or entropy is empty:',
          int(zf_confound[['mean_power', 'mean_spectral_entropy']].isna().any(axis=1).sum()))
else:
    print('skipped: REPAIR_EXPORTS is False in Cell 1')

skipped: REPAIR_EXPORTS is False in Cell 1


In [5]:
# Cell P2: rebuild the starling syllable-position table (Fig. 6D) with the bird kept
# The export held this table without a bird column, so Fig. 6D could not weight birds equally.
# Same songs, syllables and bins as the S4 analysis notebook; each row now says whose it is.
if REPAIR_EXPORTS:
    STARLING_POSITION_TABLES = {      # the per-bird syllable tables that notebook read
        '1556': 's_b1556h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
        '1484': 's_b1484h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
        '1364': 's_b1364h_25/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
        '1354': 's_b1354_25/syllable_df_labels_mindistpoint2_featureanalysismod_20250611.pickle',
        '1481': 's_b1481h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
        '1349': 's_b1349h_23/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    }

    # each bird's wav folders and model files, matched on bird number as in Cell 5
    starling_dirs, starling_models = {}, {}
    for d in sorted(x for x in glob.glob(os.path.join(DATA_ROOT, '*')) if os.path.isdir(x)):
        number = None if os.path.basename(d.rstrip('/')).startswith('zf_') else bird_num(os.path.basename(d.rstrip('/')))
        if number is None:
            continue
        regair_dir = os.path.join(d, 'processed', 'test')
        nested = os.path.join(regair_dir, 'song_regair')             # some birds keep test wavs one level down
        if os.path.isdir(nested) and len(wavs_as_scored(nested)) > 0:
            regair_dir = nested
        dirs = {'regair': regair_dir, 'heliox': os.path.join(d, 'processed', 'song_heliox')}
        n_wavs = sum(len(wavs_as_scored(p)) for p in dirs.values() if os.path.isdir(p))
        if number in starling_dirs and n_wavs == 0:
            continue                                                 # an empty folder beside the real one
        starling_dirs[number] = dirs
    for p in sorted(glob.glob(os.path.join(MODELS_ROOT, '*.pkl'))):
        for cond in COND_ORDER:
            m = re.match(r'^(?P<prefix>.*?)s4_genpred_dict_%s_full\.pkl$' % cond, os.path.basename(p))
            if m:
                number = m.group('prefix').rstrip('_') or UNPREFIXED_MODEL_BIRD
                if number.isdigit():
                    starling_models.setdefault(number, {})[cond] = p

    starling_tables, starling_position = [], []
    for number in sorted(starling_dirs):
        if set(starling_models.get(number, {})) < set(COND_ORDER):
            continue                                    # not scored in both conditions
        raw = load_pickle(os.path.join(BASE_DIR, STARLING_POSITION_TABLES[number]))
        cols = list(raw.columns)                        # by position: one table carries a column name twice
        syl = raw.iloc[:, [cols.index(c) for c in ['file', 'condition', 'start_time', 'end_time']]].reset_index(drop=True)
        del raw
        syl['filename'] = syl['file'].astype(str).apply(os.path.basename)
        syl['condition'] = syl['condition'].astype(str)
        song_rows = syl.groupby(['condition', 'filename'], sort=False).groups
        for cond in COND_ORDER:
            table, traces = scored_songs(number, cond, starling_dirs[number][cond], starling_models[number][cond],
                                         MIN_SONG_S['starling'])
            starling_tables.append(table)
            for song in table.itertuples():
                rows = song_rows.get((cond, song.filename))   # a 'regairb' syllable never matches a 'regair' song
                if rows is None:
                    continue
                for s in syl.loc[rows, ['start_time', 'end_time']].itertuples():
                    profile = position_profile(traces[song.song_idx], s.start_time, s.end_time)
                    if profile is not None:
                        starling_position.append((number, cond, profile))
            del traces
        del syl
    starling_songs = pd.concat(starling_tables, ignore_index=True)
    starling_position = position_table(starling_position)

    print('starling, per bird:')
    print(pd.DataFrame({'scored songs': starling_songs.groupby('bird_id').size(),
                        'syllables in Fig. 6D': starling_position.groupby('bird_id').size() // N_POSITION_BINS})
          .fillna(0).astype(int).to_string())
else:
    print('skipped: REPAIR_EXPORTS is False in Cell 1')

skipped: REPAIR_EXPORTS is False in Cell 1


In [6]:
# Cell P3: check the rebuilt tables against the exports on disk, then save
# Rationale: the rebuild should give back exactly what the exports already hold, plus the three
# missing zebra finches and the bird column. If it does not, the songs are no longer lined up with
# the model rows the way they were, and nothing is written.
if REPAIR_EXPORTS:
    def same_rows(new, old, columns):
        """True if the two tables hold the same rows, in any order."""
        if len(new) != len(old):
            return False
        a = new[columns].sort_values(columns).reset_index(drop=True)
        b = old[columns].sort_values(columns).reset_index(drop=True)
        for c in columns:
            if pd.api.types.is_numeric_dtype(b[c]):
                if not np.allclose(a[c].astype(float), b[c].astype(float), rtol=1e-6, atol=1e-9, equal_nan=True):
                    return False
            elif not (a[c].astype(str) == b[c].astype(str)).all():
                return False
        return True


    old_zf, old_starling = load_pickle(ZF_EXPORT_PATH), load_pickle(STARLING_EXPORT_PATH)
    had = set(old_zf['confound_df']['bird_id'].astype(str))          # the zebra finches that did match before
    song_cols = ['bird_id', 'condition', 'duration_s', 's4_avg_nll']
    cov_cols = ['bird_id', 'condition', 'mean_power', 'mean_spectral_entropy', 's4_avg_nll']
    pos_cols = ['condition', 'bin', 'nll']
    checks = {
        'zebra finch songs (all six birds)': same_rows(zf_songs, old_zf['duration_df'], song_cols),
        'zebra finch power / entropy, the three birds already there':
            same_rows(zf_confound[zf_confound['bird_id'].isin(had)], old_zf['confound_df'], cov_cols),
        'zebra finch syllable positions, the three birds already there':
            same_rows(zf_position[zf_position['bird_id'].isin(had)], old_zf['syllable_pos_df'], pos_cols),
        'starling songs': same_rows(starling_songs, old_starling['duration_df'], song_cols),
        'starling syllable positions': same_rows(starling_position, old_starling['syllable_pos_df'], pos_cols),
    }
    for name, ok in checks.items():
        print(f"  {'identical to the export' if ok else 'DIFFERENT FROM THE EXPORT':26s} {name}")
    if not all(checks.values()):
        raise RuntimeError('the rebuild does not reproduce the exports, so nothing was saved: '
                           + '; '.join(name for name, ok in checks.items() if not ok))

    # Once the exports hold the repair, a rebuild gives back exactly what is on disk. Saving again would only
    # pile up dated copies of the same file, so nothing is written in that case.
    full_pos_cols = ['bird_id'] + pos_cols
    already_repaired = (all('bird_id' in e['syllable_pos_df'].columns for e in (old_zf, old_starling))
                        and same_rows(zf_confound, old_zf['confound_df'], cov_cols)
                        and same_rows(zf_position, old_zf['syllable_pos_df'], full_pos_cols)
                        and same_rows(starling_position, old_starling['syllable_pos_df'], full_pos_cols))

    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    note = (f'rebuilt {stamp} by s4_nll_build_data.ipynb (Cells P1-P3): zebra finch syllable-table names '
            f'matched with or without z_, bird_id added to syllable_pos_df')
    new_zf = dict(old_zf, confound_df=zf_confound[cov_cols].copy(), duration_df=zf_songs[song_cols].copy(),
                  syllable_pos_df=zf_position[['bird_id'] + pos_cols].copy(), repaired=note)
    new_starling = dict(old_starling, syllable_pos_df=starling_position[['bird_id'] + pos_cols].copy(), repaired=note)
    if already_repaired:
        print('\nThe exports on disk already hold the repair, so nothing was written.')
    else:
        for path, export in [(ZF_EXPORT_PATH, new_zf), (STARLING_EXPORT_PATH, new_starling)]:
            backup = path.replace('.pkl', f'.before_{stamp}.pkl')
            shutil.copy2(path, backup)                   # the export as it was, kept beside the new one
            with open(path, 'wb') as f:
                pickle.dump(export, f)
            print(f'saved {path}\n   previous version kept as {os.path.basename(backup)}')
    print(f'\nzebra finch power / entropy table: {old_zf["confound_df"]["bird_id"].nunique()} birds, '
          f'{len(old_zf["confound_df"])} songs  ->  {zf_confound["bird_id"].nunique()} birds, {len(zf_confound)} songs')
    print(f'zebra finch syllables in Fig. 6D: {len(old_zf["syllable_pos_df"]) // N_POSITION_BINS}  ->  '
          f'{len(zf_position) // N_POSITION_BINS}')
else:
    print('skipped: REPAIR_EXPORTS is False in Cell 1')

skipped: REPAIR_EXPORTS is False in Cell 1


## 3. Tables from the export pickles

These come straight from the two export pickles written by the S4 analysis notebooks (and
repaired in Section 2), so this section is fast in either mode.

In [7]:
# Cell 4: stack the two species and build the shared covariate table
exports = {'starling': load_pickle(STARLING_EXPORT_PATH), 'zebra_finch': load_pickle(ZF_EXPORT_PATH)}


def tag_and_combine(key):
    """Stack the two species' frames, adding a species tag and a bird_uid that is unique
    across species (bird ids are not guaranteed distinct between the two exports)."""
    frames = []
    for sp, exp in exports.items():
        df = exp[key].copy()
        df['species'] = sp
        # The zebra finch export names birds by S4 folder ('zf_c8c19_24'); the audio pass (Cell 9)
        # drops the 'zf_'. Dropping it here too gives every table the same name for a bird.
        df['bird_uid'] = sp + '_' + df['bird_id'].astype(str).str.replace(r'^zf_', '', regex=True)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


region_nll = tag_and_combine('long_df')             # region: silence / song
boundary_nll = tag_and_combine('long_boundary_df')  # region: boundary / interior
confound_df = tag_and_combine('confound_df')        # mean_spectral_entropy (its mean_power is not used, see Cell L1)
duration_df = tag_and_combine('duration_df')

# Fig. 6D can average bird by bird only if both exports say which bird each syllable came from.
if all('bird_id' in exp['syllable_pos_df'].columns for exp in exports.values()):
    syllable_position_nll = tag_and_combine('syllable_pos_df')
else:
    syllable_position_nll = pd.concat(
        [exp['syllable_pos_df'].assign(species=sp) for sp, exp in exports.items()], ignore_index=True)
    print('[NOTE] syllable_pos_df has no bird_id in the export, so Fig. 6D can only pool syllables within '
          'a species. To weight birds equally, set REPAIR_EXPORTS = True in Cell 1 and run Cells P1 to P3.')

# Fig. 6I compares the condition effect before and after each covariate. All three
# covariates must sit on ONE set of songs, or the "total" effect would differ between them.
# Duration and the syllable tables' spectral entropy are joined here; amplitude (Cell L1) and BioSound's
# spectral entropy (Cell B1), which is the entropy the panel uses, are added to the same songs.
# The exports have no song id, so songs are matched on (bird, condition, song-mean NLL).
MERGE_KEYS = ['bird_uid', 'condition', 's4_avg_nll']
conf_slim = confound_df[['bird_uid', 'condition', 'species', 's4_avg_nll', 'mean_spectral_entropy']]
dur_slim = duration_df[['bird_uid', 'condition', 's4_avg_nll', 'duration_s']]
song_covariates = conf_slim.merge(dur_slim, on=MERGE_KEYS, how='inner')
if len(song_covariates) > min(len(conf_slim), len(dur_slim)):
    # more rows out than in means the float key matched one song to several
    raise RuntimeError('song-mean NLL is not a unique song key; covariate table would duplicate songs')

print('region_nll', region_nll.shape, '| boundary_nll', boundary_nll.shape,
      '| syllable_position_nll', syllable_position_nll.shape)
print(f'song_covariates: {len(song_covariates)} songs with entropy and duration '
      f'(of {len(conf_slim)} with entropy, {len(dur_slim)} with duration)')
# Where a bird goes missing: in the export's entropy table, or in the merge above.
steps = {'entropy (confound_df)': confound_df, 'duration (duration_df)': duration_df,
         'both (song_covariates)': song_covariates}
print('birds and songs at each step:')
print(pd.concat({name: t.groupby('species').agg(birds=('bird_uid', 'nunique'), songs=('bird_uid', 'size'))
                 for name, t in steps.items()}, axis=1).to_string())
any_absent = False
for name, t in steps.items():
    absent = sorted(set(region_nll['bird_uid']) - set(t['bird_uid']))
    if absent:
        any_absent = True
        print(f'  not in {name}: {absent}')
if any_absent:
    print('  Cells P1 to P3 put missing zebra finches back: set REPAIR_EXPORTS = True in Cell 1.')

require_complete(region_nll, 'region_nll', 'bird_uid', 'condition', species_col='species')
require_complete(boundary_nll, 'boundary_nll', 'bird_uid', 'condition', species_col='species')
# this table was the one left unchecked, which is how Fig. 6I came to rest on three zebra finches
require_complete(song_covariates, 'song_covariates', 'bird_uid', 'condition', species_col='species')

region_nll (4188, 6) | boundary_nll (137922, 6) | syllable_position_nll (1380720, 6)
song_covariates: 2094 songs with entropy and duration (of 2094 with entropy, 2114 with duration)
birds and songs at each step:
            entropy (confound_df)       duration (duration_df)       both (song_covariates)      
                            birds songs                  birds songs                  birds songs
species                                                                                          
starling                        6   941                      6   961                      6   941
zebra_finch                     6  1153                      6  1153                      6  1153
region_nll: complete (4188 rows)
boundary_nll: complete (137922 rows)
song_covariates: complete (2094 rows)


## 4. Tables from audio and S4 output

Skipped entirely when the three cached tables are reused.

One pass over the wavs feeds all three tables, so each model file is loaded once and
every table rests on the same wav-to-row alignment.

In [8]:
# Cell 5: find each bird's wavs and model files
# Rationale: starling model files are named by bird number only ('1349_', '1556') and one
# bird's files have no prefix, so matching is on bird number, not folder name. Two folders
# can share a number (an empty 's_b1349_23' sits beside 's_b1349h_23'); the one that holds
# wavs wins, and two folders with wavs is an error instead of a guess.
if FROM_AUDIO:
    MODEL_PATTERNS = {c: re.compile(r'^(?P<prefix>.*?)s4_genpred_dict_%s_full\.pkl$' % c)
                      for c in COND_ORDER}

    def _list_wavs(d):
        """Plain os.listdir order, deliberately unsorted (see the row manifest below)."""
        return [f for f in os.listdir(d) if f.lower().endswith('.wav')]

    registry = []
    for d in sorted(x for x in glob.glob(os.path.join(DATA_ROOT, '*')) if os.path.isdir(x)):
        folder = os.path.basename(d.rstrip('/'))
        species = 'zebra_finch' if folder.startswith('zf_') else 'starling'
        bird_id = folder[len('zf_'):] if species == 'zebra_finch' else bird_num(folder)
        processed = os.path.join(d, 'processed')
        regair_dir = os.path.join(processed, 'test')
        nested = os.path.join(regair_dir, 'song_regair')   # some birds keep test wavs one level down
        if os.path.isdir(nested) and len(_list_wavs(nested)) > 0:
            regair_dir = nested
        wav_dir = {'regair': regair_dir, 'heliox': os.path.join(processed, 'song_heliox')}
        n_wavs = {c: len(_list_wavs(p)) if os.path.isdir(p) else 0 for c, p in wav_dir.items()}
        registry.append(dict(folder=folder, species=species, bird_id=bird_id, wav_dir=wav_dir,
                             n_wavs=n_wavs, pkl={}))

    def folder_for_prefix(prefix):
        p = prefix.rstrip('_') or UNPREFIXED_MODEL_BIRD
        matches = [r for r in registry if p in (r['folder'], r['bird_id'])]
        with_data = [r for r in matches if sum(r['n_wavs'].values()) > 0]
        if len(with_data) > 1:
            raise ValueError(f'prefix "{prefix}" matches several folders with wavs: '
                             f'{[r["folder"] for r in with_data]}')
        return with_data[0] if with_data else None

    for path in sorted(glob.glob(os.path.join(MODELS_ROOT, '*.pkl'))):
        for cond, pat in MODEL_PATTERNS.items():
            m = pat.match(os.path.basename(path))
            if not m:
                continue
            reg = folder_for_prefix(m.group('prefix'))
            if reg is None:
                print(f'  [no folder with wavs] {os.path.basename(path)}')
                continue
            reg['pkl'][cond] = path

    birds = [r for r in registry if set(r['pkl']) >= set(COND_ORDER)]
    print(pd.DataFrame([dict(folder=r['folder'], species=r['species'], bird_id=r['bird_id'],
                             regair_wavs=r['n_wavs']['regair'], heliox_wavs=r['n_wavs']['heliox'],
                             regair_model=os.path.basename(r['pkl']['regair']),
                             heliox_model=os.path.basename(r['pkl']['heliox']))
                        for r in birds]).to_string(index=False))
    found = pd.Series([r['species'] for r in birds]).value_counts().to_dict()
    if found != EXPECTED_BIRDS:
        message = f'found {found} birds with both model files, expected {EXPECTED_BIRDS}'
        if not ALLOW_INCOMPLETE:
            raise RuntimeError(message + '. Set ALLOW_INCOMPLETE = True in Cell 2 to continue anyway.')
        INCOMPLETE.append(message)
        print('[INCOMPLETE]', message)
else:
    print('skipped: reusing cached tables')

skipped: reusing cached tables


In [9]:
# Cell 6: the row manifest
# Rationale: row i of a model's logprobs array is the i-th wav in the order the scoring
# notebook listed the folder, which was plain os.listdir. That order belongs to the
# filesystem, not to the data: copy the folder and it can change, and every NLL trace then
# attaches to the wrong song with no error (the row COUNT still matches). So the first
# rebuild records the order in a small csv and every later rebuild reads it back.
# The manifest assumes the order on disk today is still the order at scoring time, which
# is the same assumption the analysis has made on every run so far.
if FROM_AUDIO:
    if os.path.exists(MANIFEST_PATH):
        manifest = pd.read_csv(MANIFEST_PATH, dtype={'folder': str, 'condition': str, 'wav_file': str})
        print(f'row manifest: {len(manifest)} rows read from {os.path.basename(MANIFEST_PATH)}')
    else:
        manifest = pd.DataFrame(columns=['folder', 'condition', 'row', 'wav_file'])
        print('row manifest: none yet, it will be recorded from the current folder listing')

    def wavs_in_row_order(reg, cond):
        listed = _list_wavs(reg['wav_dir'][cond])
        known = manifest[(manifest['folder'] == reg['folder']) & (manifest['condition'] == cond)]
        if known.empty:
            return listed
        known = known.sort_values('row')['wav_file'].tolist()
        if set(known) != set(listed):
            raise RuntimeError(
                f'{reg["folder"]}/{cond}: wavs on disk differ from the row manifest '
                f'({len(set(listed) - set(known))} new, {len(set(known) - set(listed))} missing). '
                f'The model rows can no longer be matched to files.')
        return known
else:
    print('skipped: reusing cached tables')

skipped: reusing cached tables


In [10]:
# Cell 7: syllable tables that the NLL traces are aligned to
if FROM_AUDIO:
    # ---- starlings: one augmented syllable table per bird ---------------------------------
    def _dedupe_columns(cols):
        seen, out = {}, []
        for c in cols:
            c0 = str(c)
            if c0 not in seen:
                seen[c0] = 1
                out.append(c0)
            else:
                seen[c0] += 1
                out.append(f'{c0}__{seen[c0]}')
        return out

    _POSIX_WRAPPER = re.compile(r"PosixPath\(['\"](.+?)['\"]\)")

    def _as_path_str(p):
        if p is None or (isinstance(p, float) and pd.isna(p)):
            return None
        s = str(p)
        m = _POSIX_WRAPPER.search(s)
        return m.group(1) if m else s

    frames = []
    for bird in STARLING_BIRDS:
        pkl = os.path.join(STARLING_SYL_DIR, bird, f'{bird}_syllable_df_augmented.pkl')
        if not os.path.exists(pkl):
            raise FileNotFoundError(f'syllable table for {bird} not found: {pkl}')
        df = load_pickle(pkl)
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)
        if not df.columns.is_unique:
            df.columns = _dedupe_columns(df.columns)
        df = df.copy()
        df['bird'] = bird                      # never fall back on indv
        df['bird_num'] = bird_num(bird)
        df['condition'] = df['condition'].astype(str).str.lower()
        df['cond2'] = np.where(df['condition'].str.contains('heliox'), 'heliox', 'regair')
        df['file'] = df['file'].apply(_as_path_str).astype(str)
        df['basename'] = df['file'].apply(os.path.basename)
        if BOUTCOL not in df.columns:
            df[BOUTCOL] = df['file']
        if POSCOL not in df.columns:
            df = df.sort_values([BOUTCOL, 'onsets_ms']).copy()
            df[POSCOL] = df.groupby(BOUTCOL).cumcount()
        frames.append(df)
    syl_raw = pd.concat(frames, ignore_index=True)

    # onsets_ms / offsets_ms are misnamed in the starling tables and hold SECONDS.
    # This checks rather than assumes, since the zebra finch tables really are in ms.
    med_off = float(pd.to_numeric(syl_raw['offsets_ms'], errors='coerce').median())
    scale = 1.0 if med_off < 60 else 1e-3
    print(f'starling median offsets_ms = {med_off:.3f} -> treating as '
          f'{"SECONDS" if scale == 1.0 else "MILLISECONDS"}')
    syl_raw['onset_s'] = pd.to_numeric(syl_raw['onsets_ms'], errors='coerce') * scale
    syl_raw['offset_s'] = pd.to_numeric(syl_raw['offsets_ms'], errors='coerce') * scale

    keep_cols = ['bird', 'bird_num', 'cond2', 'file', 'basename', BOUTCOL, POSCOL,
                 'onset_s', 'offset_s', LABCOL]
    starling_syl = syl_raw[keep_cols].dropna(subset=['onset_s', 'offset_s', LABCOL]).copy()
    starling_syl = starling_syl.sort_values(['bird', BOUTCOL, POSCOL]).reset_index(drop=True)
    print(f'starling syllables: {len(starling_syl)} usable of {len(syl_raw)}, '
          f'{starling_syl["bird"].nunique()} birds')
    del syl_raw, frames

    # ---- zebra finches: one combined table, times in seconds within each song wav ----------
    zf_syl = pd.read_pickle(ZF_SYL_PATH)
    zf_syl['heliox'] = zf_syl['heliox'].astype(bool)
    zf_syl['_song'] = zf_syl['song_id'].astype(str)
    # grouped once, instead of filtering the whole table for every song
    zf_song_groups = zf_syl.groupby(['indv', 'heliox', '_song'], sort=False, observed=True).groups

    def zf_indv_for(folder):
        """'zf_c8c19_24' -> the matching name in the syllable table, with or without 'z_'."""
        candidate = folder.replace('zf_', '', 1)
        known = set(zf_syl['indv'].unique())
        for name in (candidate, 'z_' + candidate):
            if name in known:
                return name
        message = f'{folder}: no bird named {candidate} or z_{candidate} in the zebra finch syllable table'
        if not ALLOW_INCOMPLETE:
            raise KeyError(message + '. Set ALLOW_INCOMPLETE = True in Cell 2 to continue without its syllables.')
        print('[INCOMPLETE]', message)
        return None

    print(f'zebra finch syllables: {len(zf_syl)}, {zf_syl["indv"].nunique()} birds')
else:
    print('skipped: reusing cached tables')

skipped: reusing cached tables


In [11]:
# Cell 8: metric definitions for the per-song trace measures (S8 C-D)
def envelope_from_audio(x, sr, smooth_ms=ENV_SMOOTH_MS):
    """Hilbert envelope of the audio, lightly smoothed. Same sample rate as the NLL trace."""
    env = np.abs(hilbert(x))
    win_n = max(3, int(round(smooth_ms / 1000.0 * sr)))
    if win_n % 2 == 0:
        win_n += 1
    w = np.hanning(win_n)
    w /= w.sum() + 1e-12
    return np.convolve(env, w, mode='same')


def _robust_sd(y):
    """MAD-based SD. NLL has no natural unit shared across birds, so the jump threshold has
    to be relative to each trace."""
    return 1.4826 * np.median(np.abs(y - np.median(y))) + 1e-12


def jump_rate(nll_binned):
    """Fraction of frame-to-frame |changes| above JUMP_THRESH_STD robust SDs. Whole trace,
    silence included."""
    if len(nll_binned) < 3:
        return np.nan
    return float(np.mean(np.abs(np.diff(nll_binned)) > JUMP_THRESH_STD * _robust_sd(nll_binned)))


def _song_runs(mask, min_run_len=3):
    """(start, end) of each stretch of True at least min_run_len bins long."""
    runs, in_run, start = [], False, None
    for i, v in enumerate(mask):
        if v and not in_run:
            start, in_run = i, True
        elif not v and in_run:
            if i - start >= min_run_len:
                runs.append((start, i))
            in_run = False
    if in_run and len(mask) - start >= min_run_len:
        runs.append((start, len(mask)))
    return runs


def peak_song_nll(nll_binned, env_binned):
    """Maximum NLL over bins that count as song (envelope above the silence threshold)."""
    runs = _song_runs(env_binned >= SILENCE_FRAC_OF_PEAK * env_binned.max())
    if not runs:
        return np.nan
    return float(np.max(np.concatenate([nll_binned[s:e] for s, e in runs])))


print('metrics ready')

metrics ready


In [12]:
# Cell 9: one pass over every wav
# For each bird and condition: load the model output once, then for each song
#   - both species:  peak song NLL, jump rate            -> song_trace_metrics
#   - starlings:     keep the 5 ms binned trace           -> joined to syllables in Cell 10
#   - zebra finches: mean NLL in each syllable and gap    -> zf_syllable_nll
if FROM_AUDIO:
    zf_edge = int(ZF_EDGE_MS / 1000 * SAMPLE_RATE)
    starling_traces = {}                      # (bird_num, condition, wav_file) -> binned trace
    song_rows, zf_rows, manifest_rows = [], [], []

    for reg in birds:
        species, min_len = reg['species'], MIN_SONG_S[reg['species']] * SAMPLE_RATE
        indv = zf_indv_for(reg['folder']) if species == 'zebra_finch' else None
        n_kept = {}
        for cond in COND_ORDER:
            files = wavs_in_row_order(reg, cond)
            arr = load_pickle(reg['pkl'][cond])['logprobs_array_full']
            if len(files) != arr.shape[0]:
                # without equal counts there is no way to say which row is which song
                raise RuntimeError(f'{reg["folder"]}/{cond}: {len(files)} wavs but {arr.shape[0]} model rows')
            n_kept[cond] = 0
            for i, wf in enumerate(files):
                manifest_rows.append(dict(folder=reg['folder'], condition=cond, row=i, wav_file=wf))
                sr, x = read_wav_mono_float(os.path.join(reg['wav_dir'][cond], wf))
                true_len = min(len(x), arr.shape[1])     # the rest of the row is padding

                # ---- zebra finch syllables and gaps, at full sample resolution -------------
                if species == 'zebra_finch':
                    idx = zf_song_groups.get((indv, cond == 'heliox', os.path.splitext(wf)[0]))
                    if idx is not None:
                        s = zf_syl.loc[idx].sort_values('start_time')
                        on = [int(t * SAMPLE_RATE) for t in s['start_time']]
                        off = [int(t * SAMPLE_RATE) for t in s['end_time']]
                        labels = s['label'].tolist()
                        trace = arr[i]
                        for j in range(len(s)):
                            # too short to trim 10 ms from both ends, or runs past the trace
                            if off[j] - on[j] < zf_edge * 3 or off[j] > trace.shape[0]:
                                continue
                            interior = trace[on[j] + zf_edge: off[j] - zf_edge].mean()
                            if np.isnan(interior):
                                continue
                            gap = np.nan
                            if j + 1 < len(s) and on[j + 1] > off[j] and on[j + 1] <= trace.shape[0]:
                                gap = trace[off[j]: on[j + 1]].mean()
                            zf_rows.append(dict(bird_id=reg['folder'], condition=cond,
                                                song_id=os.path.splitext(wf)[0], label=labels[j],
                                                is_intro=labels[j] == 0,
                                                interior_nll=interior, gap_nll=gap))

                # ---- song-level measures on the binned trace ---------------------------------
                if true_len < min_len:
                    continue
                nll_b = bin_trace(arr[i, :true_len], BIN_SAMPLES)
                if len(nll_b) < 5:
                    continue
                env = envelope_from_audio(x, sr)[:true_len]
                song_rows.append(dict(
                    bird=reg['bird_id'], species=species, condition=cond, wav_file=wf,
                    nll_p100=peak_song_nll(nll_b, bin_trace(env, BIN_SAMPLES)),
                    jump_rate_raw=jump_rate(nll_b),
                    # The next two come from the song after it was rescaled to its own peak
                    # (read_wav_mono_float), so they say nothing about how loud it was. They stay so
                    # that a rebuild still matches the cache, and Cell 13 does not save them. The
                    # amplitude used in the paper is level_db, added in Cell L1.
                    power=float(np.mean(env ** 2)),        # unbinned, native resolution
                    amplitude=float(np.mean(env))))
                if species == 'starling':
                    starling_traces[(reg['bird_id'], cond, wf)] = dict(nll=nll_b, dur_s=true_len / SAMPLE_RATE)
                n_kept[cond] += 1
            del arr
        print(f'{reg["folder"]:14s} {species:12s} songs kept: {n_kept["regair"]} regair, {n_kept["heliox"]} heliox')

    song_trace_metrics = pd.DataFrame(song_rows)
    song_trace_metrics['bird_uid'] = song_trace_metrics['species'] + '_' + song_trace_metrics['bird'].astype(str)
    zf_syllable_nll = pd.DataFrame(zf_rows)
    zf_syllable_nll['is_heliox'] = (zf_syllable_nll['condition'] == 'heliox').astype(int)
    zf_syllable_nll['is_intro_num'] = zf_syllable_nll['is_intro'].astype(int)
    pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False)
    print(f'\n{len(song_trace_metrics)} songs, {len(zf_syllable_nll)} zebra finch syllables; '
          f'row manifest saved ({len(manifest_rows)} rows)')
else:
    print('skipped: reusing cached tables')

skipped: reusing cached tables


In [13]:
# Cell 10: starling per-syllable and per-gap NLL
# Each syllable's NLL is the mean of the binned trace over its onset-to-offset window; each
# gap's is the mean from that syllable's offset to the next onset, within the same bout.
if FROM_AUDIO:
    def window_mean(trace, t0, t1):
        i0 = max(0, int(np.floor(t0 * 1000.0 / BIN_MS)))
        i1 = min(len(trace), int(np.ceil(t1 * 1000.0 / BIN_MS)))
        if i1 - i0 < MIN_BINS_PER_WINDOW:
            return np.nan
        return float(np.mean(trace[i0:i1]))

    rows, n_songs = [], 0
    for (bird, cond, base), g in starling_syl.groupby(['bird', 'cond2', 'basename'], sort=False):
        bnum = g['bird_num'].iloc[0]
        entry = starling_traces.get((bnum, cond, base))
        if entry is None:
            continue           # song not scored by S4 (training song), or under the length cutoff
        trace = entry['nll']
        n_songs += 1
        song_median, song_mean, song_peak = float(np.median(trace)), float(np.mean(trace)), float(np.max(trace))
        g = g.sort_values(POSCOL)
        on, off = g['onset_s'].to_numpy(), g['offset_s'].to_numpy()
        lab, pos, bout = g[LABCOL].to_numpy(), g[POSCOL].to_numpy(), g[BOUTCOL].to_numpy()
        for i in range(len(g)):
            nll_gap = nll_pair = lab_to = np.nan
            if i + 1 < len(g) and bout[i] == bout[i + 1]:      # transition i -> i+1, same bout only
                lab_to = lab[i + 1]
                if on[i + 1] > off[i]:
                    nll_gap = window_mean(trace, off[i], on[i + 1])
                nll_pair = window_mean(trace, on[i], off[i + 1])
            rows.append({
                'bird': bird, 'bird_num': bnum, 'cond2': cond, 'basename': base,
                'bout': bout[i], 'pos': pos[i], 'label': lab[i], 'label_to': lab_to,
                'onset_s': on[i], 'offset_s': off[i], 'dur_s': off[i] - on[i],
                'nll_syl': window_mean(trace, on[i], off[i]), 'nll_gap': nll_gap, 'nll_pair': nll_pair,
                'song_median_nll': song_median, 'song_mean_nll': song_mean,
                'song_peak_nll': song_peak, 'song_dur_s': entry['dur_s']})
    starling_syllable_nll = pd.DataFrame(rows)
    starling_syllable_nll['bird_uid'] = starling_syllable_nll['bird']
    print(f'{n_songs} songs joined, {len(starling_syllable_nll)} syllable rows')
else:
    print('skipped: reusing cached tables')

skipped: reusing cached tables


In [14]:
# Cell 11: compare a rebuild against the existing cache, then save
# Rationale: the cached tables are what the paper's numbers were computed from. A rebuild
# should reproduce them. If it does not, the old file is kept beside the new one with a
# timestamp so the two can be compared, and the difference is printed here.
def compare_to_cache(name, new):
    path = CACHE_FILES[name]
    if not os.path.exists(path):
        print(f'{name}: no earlier cache to compare against')
        return
    old = pd.read_pickle(path)
    shared = [c for c in old.columns if c in new.columns]
    notes = []
    if len(old) != len(new):
        notes.append(f'row count {len(old)} -> {len(new)}')
    else:
        for c in shared:
            a, b = old[c].reset_index(drop=True), new[c].reset_index(drop=True)
            if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
                a, b = a.astype(float), b.astype(float)
                if not np.allclose(a, b, rtol=1e-9, atol=1e-12, equal_nan=True):
                    notes.append(f'{c}: max |difference| {np.nanmax(np.abs(a - b)):.3g}')
            elif not (a.astype(str) == b.astype(str)).all():
                notes.append(f'{c}: {int((a.astype(str) != b.astype(str)).sum())} rows differ')
    if not notes:
        print(f'{name}: rebuild is identical to the cache on {len(shared)} shared columns, {len(new)} rows')
        return
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    backup = path.replace('.pkl', f'.before_{stamp}.pkl')
    shutil.copy2(path, backup)
    print(f'[DIFFERENT] {name}: ' + '; '.join(notes))
    print(f'            previous cache kept as {os.path.basename(backup)}')


if FROM_AUDIO:
    tables = {'starling_syllable_nll': starling_syllable_nll, 'zf_syllable_nll': zf_syllable_nll,
              'song_trace_metrics': song_trace_metrics}
    # checked before anything is written, so an incomplete table never reaches disk
    require_complete(starling_syllable_nll, 'starling_syllable_nll', 'bird', 'cond2', species='starling')
    require_complete(zf_syllable_nll, 'zf_syllable_nll', 'bird_id', 'condition', species='zebra_finch')
    require_complete(song_trace_metrics, 'song_trace_metrics', 'bird_uid', 'condition', species_col='species')
    for name, df in tables.items():
        compare_to_cache(name, df)
        df.to_pickle(CACHE_FILES[name])
else:
    starling_syllable_nll = pd.read_pickle(CACHE_FILES['starling_syllable_nll'])
    zf_syllable_nll = pd.read_pickle(CACHE_FILES['zf_syllable_nll'])
    song_trace_metrics = pd.read_pickle(CACHE_FILES['song_trace_metrics'])
    require_complete(starling_syllable_nll, 'starling_syllable_nll', 'bird', 'cond2', species='starling')
    require_complete(zf_syllable_nll, 'zf_syllable_nll', 'bird_id', 'condition', species='zebra_finch')
    require_complete(song_trace_metrics, 'song_trace_metrics', 'bird_uid', 'condition', species_col='species')

starling_syllable_nll: complete (28057 rows)
zf_syllable_nll: complete (34051 rows)
song_trace_metrics: complete (2114 rows)


## 5. k-gram sets

The retained k-grams come from the k-gram notebook's export and cannot be rebuilt here,
because the selection depends on the Markov fold-enrichment null. This section checks
that every bird in the NLL table either has k-gram sets under the same name or is on the
declared exclusion list. A bird in neither group used to drop out without a message.

In [15]:
# Cell 12: load the k-gram sets and check bird names against the NLL table
motif_export = load_pickle(MOTIF_PKL)
if motif_export.get('LABCOL') not in (None, LABCOL):
    raise RuntimeError(f'k-gram sets were built on {motif_export.get("LABCOL")}, the NLL table uses {LABCOL}')

nll_birds = set(starling_syllable_nll['bird'].unique())
motif_birds = {b for (b, _, _) in motif_export['final_motif_sets']}

leaking = set(KGRAM_NLL_EXCLUDE) & motif_birds
if leaking:
    raise RuntimeError(f'{sorted(leaking)} is excluded but has k-gram sets under that name. '
                       f'Those sets use a different clustering; relabel before including it.')
missing = nll_birds - set(KGRAM_NLL_EXCLUDE) - motif_birds
if missing:
    raise RuntimeError(f'no k-gram sets for {sorted(missing)}; check bird names in {os.path.basename(MOTIF_PKL)}')

kgram_birds = sorted(nll_birds - set(KGRAM_NLL_EXCLUDE))
# plain Python ints and strs only, so the pickle does not depend on the numpy version
starling_kgram_sets = {
    (str(b), str(c), int(k)): {tuple(int(x) for x in seq) for seq in seqs}
    for (b, c, k), seqs in motif_export['final_motif_sets'].items() if b in kgram_birds}

print(f'k-gram NLL analysis: {len(kgram_birds)} birds {kgram_birds}')
print(f'excluded on purpose: {KGRAM_NLL_EXCLUDE}')
print(f'{len(starling_kgram_sets)} (bird, condition, k) sets; k = {sorted({k for (_, _, k) in starling_kgram_sets})}; '
      f'selection: prevalence >= {motif_export.get("THRESHOLD")}, fold >= {motif_export.get("FOLD_CUT_FINAL")}')

k-gram NLL analysis: 5 birds ['s_b1349h_23', 's_b1354_25', 's_b1481h_24', 's_b1484h_24', 's_b1556h_24']
excluded on purpose: {'s_b1364_25': 'k-gram sets come from a different clustering (s_b1364h_25)'}
60 (bird, condition, k) sets; k = [3, 4, 5, 6, 7, 8]; selection: prevalence >= 0.01, fold >= 2


## 6. Song amplitude and spectral entropy

**Cell L1** adds the amplitude of every song to the two song tables: `song_covariates` (Fig. 6I) and
`song_trace_metrics` (S8 C-D). Songs recorded in regular air after heliox and songs with no
syllables in the syllable tables are taken out of `song_trace_metrics` here, so that S8 C-D
rests on the same songs as Fig. 6I. The cell prints how many of each and names the second kind.

**Cell B1** adds BioSound's spectral entropy to `song_covariates` (Fig. 6I). It first checks that
BioSound and the syllable tables hold the same syllables for each song, and it stops without
changing anything if a bird's songs do not line up or if any song would be left without a value.

In [16]:
# Cell L1: amplitude of every S4 song (for Fig. 6I and S8 C-D)
# Run after Cells 4 and 11 and before Cell 13. Needs syllable_level_data.pkl from data_figs Cell A1.
# The two "power" covariates did not measure how loud a song was: mean_power is the mean of a spectrogram
# image rescaled syllable by syllable, and power is the envelope of a song rescaled to its own peak. This
# gives every S4 song its amplitude: the mean over its syllables of the RMS amplitude of the waveform as
# recorded (dB re full scale). The columns keep the names they have in the amplitude file:
#   song_covariates['mean_level_db']   -> Fig. 6I
#   song_trace_metrics['level_db']     -> S8 C-D
# Two kinds of scored song are taken out of S8 C-D here, so that it rests on the same songs as Fig. 6I:
# songs recorded in regular air after heliox (regairb), which the rest of the paper leaves out, and songs
# with no syllables in the syllable tables, which have no amplitude to adjust for.
LEVEL_PATH = _path('HELIOX_SYLLABLE_LEVEL_DATA', os.path.join(DERIVED, 'syllable_level_data.pkl'))
SONG_KEY_PATH = os.path.join(CACHE_DIR, 's4_song_key.csv')     # each S4 song's wav name and where it sits in the amplitude table
ZF_FULL_NAME = {'p7y2': 'z_p7y2_23', 'r11g11': 'z_r11g11_24', 'y11c13': 'z_y11c13_24'}   # bird names in the amplitude table


def bird_key(species, bird):
    """One name per bird whichever table it comes from: 'zf_c8c19_24', 'c8c19_24' and
    'zebra_finch_zf_c8c19_24' all become 'zebra_finch_c8c19_24'."""
    b = re.sub(r'^(starling|zebra_finch)_', '', str(bird))
    return species + '_' + re.sub(r'^zf_', '', b)


# ---- which wav each scored song is, and its song-mean NLL (the only handle song_covariates has) -----
if os.path.exists(SONG_KEY_PATH):
    song_key = pd.read_csv(SONG_KEY_PATH, dtype={'wav_file': str, 'level_bird': str, 'level_day': str})
    print(f'song names: {len(song_key)} songs read from {os.path.basename(SONG_KEY_PATH)}')
else:
    if not all(name in globals() for name in ('starling_songs', 'zf_songs', 'zf_syl')):
        raise RuntimeError('The first run of this cell needs the song names that Cells P1 and P2 make. Set '
                           'REPAIR_EXPORTS = True in Cell 1, run Cells P1 to P3, then this cell again. It saves '
                           'the names, so later runs do not need them.')
    st = starling_songs.assign(species='starling', wav_file=starling_songs['filename'].astype(str))
    # a zebra finch song is one bout of one recording day; the amplitude table names it that way
    place = (zf_syl.assign(indv=zf_syl['indv'].astype(str), song_id=zf_syl['song_id'].astype(str), sess=zf_syl['sess'].astype(str))
             .groupby(['indv', 'song_id'], sort=False)
             .agg(level_day=('sess', 'first'), level_bout=('bout_index', 'first'),
                  n_days=('sess', 'nunique'), n_bouts=('bout_index', 'nunique')).reset_index())
    if ((place['n_days'] != 1) | (place['n_bouts'] != 1)).any():
        raise RuntimeError('some zebra finch song_ids span more than one day or bout, so they cannot be matched to the amplitude table')
    zs = zf_songs.assign(indv=zf_songs['indv'].astype(str), song_id=zf_songs['song_id'].astype(str)) \
                 .merge(place[['indv', 'song_id', 'level_day', 'level_bout']], on=['indv', 'song_id'], how='left')
    zs = zs.assign(species='zebra_finch', wav_file=zs['filename'].astype(str),
                   level_bird=zs['indv'].map(lambda b: ZF_FULL_NAME.get(b, b)))
    song_key = pd.concat([st, zs], ignore_index=True)
    song_key['bird'] = [bird_key(s, b) for s, b in zip(song_key['species'], song_key['bird_id'])]
    song_key['level_bout'] = pd.to_numeric(song_key['level_bout'], errors='coerce')
    song_key = song_key[['species', 'bird', 'condition', 'wav_file', 's4_avg_nll', 'level_bird', 'level_day', 'level_bout']]
    song_key.to_csv(SONG_KEY_PATH, index=False)
    print(f'song names: {len(song_key)} songs taken from Cells P1 and P2 and saved to {os.path.basename(SONG_KEY_PATH)}')
if song_key.duplicated(['species', 'wav_file']).any():
    raise RuntimeError('a wav name appears twice among the scored songs, so songs cannot be matched by name')

# ---- amplitude of each song: the mean over its syllables ---------------------------------------------
level_file = load_pickle(LEVEL_PATH)
lv = level_file['syllables']
# how the amplitudes were measured (band, filter), carried into the saved file by Cell 13
AMPLITUDE_INFO = {'file': os.path.basename(LEVEL_PATH),
                  'created': level_file.get('meta', {}).get('created'),
                  'settings': level_file.get('meta', {}).get('settings')}
lv = lv[lv['level_db'].notna()].astype({'species': str, 'bird': str, 'condition': str, 'day': str, 'song': str})
per_song = dict(level_db=('level_db', 'mean'), level_condition=('condition', lambda c: c.mode().iat[0]),
                level_syllables=('level_db', 'size'))
st_level = lv[lv['species'] == 'starling'].groupby('song').agg(**per_song).reset_index().rename(columns={'song': 'wav_file'})
zl = lv[lv['species'] == 'zebra_finch']
zl = zl.assign(level_bout=pd.to_numeric(zl['song'].str.extract(r'_bout(.+)$', expand=False), errors='coerce'))
zf_level = zl.groupby(['bird', 'day', 'level_bout']).agg(**per_song).reset_index() \
             .rename(columns={'bird': 'level_bird', 'day': 'level_day'})
song_level = pd.concat([
    song_key[song_key['species'] == 'starling'].merge(st_level, on='wav_file', how='left'),
    song_key[song_key['species'] == 'zebra_finch'].merge(zf_level, on=['level_bird', 'level_day', 'level_bout'], how='left'),
], ignore_index=True)
# an amplitude counts only if the amplitude table puts the song in the condition S4 scored it under
song_level['level_status'] = np.where(song_level['level_condition'].isna(), 'no syllables in the amplitude table',
                             np.where(song_level['level_condition'] == song_level['condition'], 'ok',
                             np.where(song_level['level_condition'] == 'regairb', 'regairb', 'condition differs')))
song_level.loc[song_level['level_status'] != 'ok', 'level_db'] = np.nan
print('\nscored songs by what the amplitude table says about them:')
print(song_level.groupby(['species', 'level_status']).size().unstack(fill_value=0).to_string())
AMPLITUDE_INFO['scored_songs_without_amplitude'] = {
    status: int(n) for status, n in song_level['level_status'].value_counts().items() if status != 'ok'}

# ---- S8 C-D: by wav name ---------------------------------------------------------------------------
before = len(song_trace_metrics)
song_trace_metrics = song_trace_metrics.drop(columns=['level_db', 'level_status'], errors='ignore').merge(
    song_level[['species', 'wav_file', 'level_db', 'level_status']], on=['species', 'wav_file'], how='left')
if len(song_trace_metrics) != before:
    raise RuntimeError('matching by wav name changed the number of songs in song_trace_metrics')
LEFT_OUT = {'regairb': 'recorded in regular air after heliox',
            'no syllables in the amplitude table': 'with no syllables in the syllable tables'}
for status, why in LEFT_OUT.items():
    rows = song_trace_metrics[song_trace_metrics['level_status'] == status]
    if len(rows):
        print(f'\nS8 C-D: {len(rows)} songs {why} are left out:', rows.groupby('bird').size().to_dict())
        if status != 'regairb':
            for name in rows['wav_file']:          # few enough to list, so they can be looked up
                print('     ', name)
song_trace_metrics = song_trace_metrics[~song_trace_metrics['level_status'].isin(list(LEFT_OUT))] \
    .drop(columns='level_status').reset_index(drop=True)

# ---- Fig. 6I: by bird, condition and song-mean NLL -------------------------------------------------
# song_covariates carries no song name. Its s4_avg_nll is the same number as in song_key, so each of its
# songs is the scored song of that bird and condition with the nearest value, and the gap must be ~0.
cov_bird = np.array([bird_key(s, b) for s, b in zip(song_covariates['species'], song_covariates['bird_uid'])])
values = song_covariates['s4_avg_nll'].to_numpy(float)
match = np.full(len(song_covariates), -1)
groups = pd.DataFrame({'bird': cov_bird, 'condition': song_covariates['condition'].astype(str).to_numpy()}).groupby(['bird', 'condition']).indices
for (b, c), idx in groups.items():
    cand = song_level[(song_level['bird'] == b) & (song_level['condition'] == c)].sort_values('s4_avg_nll')
    if cand.empty:
        continue
    u, v = cand['s4_avg_nll'].to_numpy(float), values[idx]
    right = np.clip(np.searchsorted(u, v), 0, len(u) - 1)        # nearest candidate on either side of each value
    left = np.clip(right - 1, 0, len(u) - 1)
    j = np.where(np.abs(u[left] - v) <= np.abs(u[right] - v), left, right)
    close = np.abs(u[j] - v) <= 1e-6 * np.abs(v) + 1e-9
    match[idx[close]] = cand.index.to_numpy()[j[close]]
if (match < 0).any() or pd.Series(match).duplicated().any():
    raise RuntimeError(f'{int((match < 0).sum())} songs of song_covariates were not found among the scored songs, and '
                       f'{int(pd.Series(match[match >= 0]).duplicated().sum())} scored songs were matched twice. '
                       'Delete s4_song_key.csv, set REPAIR_EXPORTS = True in Cell 1, run Cells P1 to P3, then this cell again.')
song_covariates = song_covariates.assign(mean_level_db=song_level.loc[match, 'level_db'].to_numpy())

for name, table, column in [('song_covariates', song_covariates, 'mean_level_db'), ('song_trace_metrics', song_trace_metrics, 'level_db')]:
    missing = table[table[column].isna()]
    if len(missing):
        message = (f'{name}: {len(missing)} songs have no amplitude: '
                   f'{missing.groupby(["species", "bird_uid"]).size().to_dict()}')
        if not ALLOW_INCOMPLETE:
            raise RuntimeError(message + '. Set ALLOW_INCOMPLETE = True in Cell 2 to continue anyway.')
        INCOMPLETE.append(message)
        print('[INCOMPLETE]', message)
    by_bird = table.groupby(['species', 'bird_uid', 'condition'])[column].mean().unstack('condition')
    diff = (by_bird['regair'] - by_bird['heliox']).groupby('species').agg(['mean', 'sem', 'size'])
    print(f'\n{name}: {len(table)} songs; song amplitude, Regair minus Heliox (dB), mean and SEM across birds:')
    print(diff.round(2).to_string())
require_complete(song_covariates, 'song_covariates', 'bird_uid', 'condition', species_col='species')
require_complete(song_trace_metrics, 'song_trace_metrics', 'bird_uid', 'condition', species_col='species')

song names: 2114 songs read from s4_song_key.csv

scored songs by what the amplitude table says about them:
level_status  no syllables in the amplitude table    ok  regairb
species                                                         
starling                                        5   941       15
zebra_finch                                     0  1153        0

S8 C-D: 15 songs recorded in regular air after heliox are left out: {'1354': 15}

S8 C-D: 5 songs with no syllables in the syllable tables are left out: {'1354': 1, '1364': 4}
      s_b1354_25_2025-03-23_11-01-01-02_55495_67590_song_regair.wav
      s_b1364h_25_2025-03-08_09-00-01-01_1033975_1071155_song_regair.wav
      s_b1364h_25_2025-03-01_11-00-01-01_1773810_1799910_song_regair.wav
      s_b1364h_25_2025-03-08_09-00-01-01_1390355_1424580_song_regair.wav
      s_b1364h_25_2025-03-03_11-00-02-02_1613205_1666790_song_heliox.wav

song_covariates: 2094 songs; song amplitude, Regair minus Heliox (dB), mean and SEM across bir

In [17]:
# Cell B1: BioSound spectral entropy (entS) of every S4 song, for Fig. 6I
# Run after Cell L1 and before Cell 13. Adds song_covariates['mean_ent_s']: the mean over a song's
# syllables of entS, the spectral entropy BioSound computes (the one shown with the other BioSound
# features in Fig. 2 and Supp. Fig. S2). mean_spectral_entropy stays in the table beside it.
# The BioSound tables are ~10 GB each, so this reads the two small files per bird that data_figs keeps:
#   biosound_paf_slim/  (Cell A4)  entS and the other 17 features, no song names
#   biosound_slim_day/  (Cell E3)  song names and onsets, with three of the features
# Both are cut from the same table in the same row order. That is checked on the three features they
# share, and only then is entS given the song name on the same row.
# BioSound and the syllable tables do not hold exactly the same syllables: in two zebra finches most songs
# have a few syllables that are in one table and not in the other. So syllables are matched one by one on
# their onset, and a song's entS is the mean over the syllables that are in both tables. That keeps the
# entropy on the same syllables as the amplitude, and leaves out what the syllable tables left out.
BIOSOUND_PAF_DIR = os.path.join(DERIVED, 'biosound_paf_slim')
BIOSOUND_DAY_DIR = os.path.join(DERIVED, 'biosound_slim_day')
BIOSOUND_BIRDS = {
    'starling': ['s_b1349h_23', 's_b1354_25', 's_b1364h_25', 's_b1481h_24', 's_b1484h_24', 's_b1556h_24'],
    'zebra_finch': ['z_c8c19_24', 'z_g14g16_24', 'z_p4g13_23', 'z_p7y2_23', 'z_r11g11_24', 'z_y11c13_24']}
SHARED_FEATURES = ['meanS', 'fund', 'F1']     # in both files
ONSET_TOLERANCE_S = 0.002                     # two onsets this close, once the clocks agree, are one syllable
MIN_SHARED = 0.5                              # a song is verified if this share of its syllables is in both tables
MIN_VERIFIED = 0.90                           # share of a bird's songs that must be verified


def song_stem(p):
    """'/a/b/song.wav' or "PosixPath('/a/b/song.wav')" -> 'song'"""
    return os.path.splitext(os.path.basename(str(p).rstrip("')\"")))[0]


def day_text(v, date_only):
    """A recording day as text. date_only keeps just the date inside it ('2025-07-30 00:00:00' -> '2025-07-30')."""
    m = re.search(r'\d{4}-\d{2}-\d{2}', str(v)) if date_only else None
    return m.group(0) if m else str(v)


def bout_text(v):
    """A bout number as text, the same whether it was stored as 7, 7.0 or '7'."""
    v = pd.to_numeric(v, errors='coerce')
    return 'none' if pd.isna(v) else str(int(v)) if float(v).is_integer() else repr(float(v))


DATE_ONLY = set()     # zebra finches whose BioSound file writes the recording day another way (filled below)


def zf_song_name(bird, day, bout):
    """A zebra finch song is one bout of one recording day."""
    return [f'{b}|{day_text(d, b in DATE_ONLY)}|{bout_text(k)}' for b, d, k in zip(bird, day, bout)]


def match_onsets(table_onsets, biosound_onsets):
    """For each syllable of a song in the syllable table, the position of the same syllable among the
    BioSound onsets of that song, or -1 if BioSound does not have it.
    The two tables need not keep the same clock, so the shift between them is found first: it is the
    difference that the largest number of onset pairs share."""
    a, b = np.asarray(table_onsets, float), np.asarray(biosound_onsets, float)
    found = np.full(len(a), -1)
    ia, ib = np.flatnonzero(np.isfinite(a)), np.flatnonzero(np.isfinite(b))
    if len(ia) == 0 or len(ib) == 0:
        return found
    d = (b[ib][None, :] - a[ia][:, None]).ravel()
    steps = np.round(d / ONSET_TOLERANCE_S).astype(np.int64)
    values, counts = np.unique(steps, return_counts=True)
    shift = np.median(d[np.abs(steps - values[counts.argmax()]) <= 1])
    order = ib[np.argsort(b[ib])]                       # BioSound positions, by onset
    sorted_b = b[order] - shift
    right = np.clip(np.searchsorted(sorted_b, a[ia]), 0, len(order) - 1)
    left = np.clip(right - 1, 0, len(order) - 1)
    nearest = np.where(np.abs(sorted_b[left] - a[ia]) <= np.abs(sorted_b[right] - a[ia]), left, right)
    close = np.abs(sorted_b[nearest] - a[ia]) <= ONSET_TOLERANCE_S
    _, first = np.unique(nearest[close], return_index=True)     # a BioSound syllable is used once
    keep = np.flatnonzero(close)[first]
    found[ia[keep]] = order[nearest[keep]]
    return found


# ---- the scored song behind each row of song_covariates (found by Cell L1) -----------------------------
if ('mean_level_db' not in song_covariates.columns or not all(name in globals() for name in ('song_level', 'match'))
        or len(match) != len(song_covariates)):
    raise RuntimeError('run Cell L1 before this cell')
scored = song_level.loc[match].reset_index(drop=True)
if not np.allclose(scored['s4_avg_nll'], song_covariates['s4_avg_nll'], rtol=1e-6, atol=1e-9):
    raise RuntimeError('song_covariates has changed since Cell L1 ran: run Cell L1 again, then this cell')
# each bird under the name of its BioSound file: starlings by bird number, zebra finches as in the amplitude file
by_number = {re.search(r'\d{4}', b).group(0): b for b in BIOSOUND_BIRDS['starling']}
scored['bio_bird'] = np.where(scored['species'] == 'zebra_finch', scored['level_bird'].astype(str),
                              scored['bird'].str.extract(r'(\d{4})', expand=False).map(by_number))

# ---- entS of every syllable, with its song ------------------------------------------------------------
frames, misaligned = [], []
for species, birds in BIOSOUND_BIRDS.items():
    for bird in birds:
        paf_path = os.path.join(BIOSOUND_PAF_DIR, f'{bird}_paf.pickle')
        day_path = os.path.join(BIOSOUND_DAY_DIR, f'{bird}_slim.pickle')
        for path, cell in [(paf_path, 'A4'), (day_path, 'E3')]:
            if not os.path.exists(path):
                raise FileNotFoundError(f'{path} not found. Cell {cell} of data_figs writes it.')
        paf = pd.read_pickle(paf_path).reset_index(drop=True)
        day = pd.read_pickle(day_path).reset_index(drop=True)
        if len(paf) != len(day) or not all(
                np.allclose(paf[f].to_numpy(float), day[f].to_numpy(float), rtol=1e-6, atol=0, equal_nan=True)
                for f in SHARED_FEATURES):
            misaligned.append(f'{bird} ({len(paf)} and {len(day)} rows)')
            continue
        if species == 'starling':
            name = day['key'].map(song_stem).to_numpy()
        else:
            day_col = day['sess' if 'sess' in day.columns else 'date']
            mine = scored[scored['bio_bird'] == bird]
            wanted = set(zf_song_name(mine['level_bird'], mine['level_day'], mine['level_bout']))
            if len(wanted & set(zf_song_name([bird] * len(day), day_col, day['bout_index']))) < 0.5 * len(wanted):
                DATE_ONLY.add(bird)      # e.g. '2025-07-30 00:00:00' here and '2025-07-30' in the syllable table
            name = zf_song_name([bird] * len(day), day_col, day['bout_index'])
        one = pd.DataFrame({'bio_bird': bird, 'song_name': name,
                            'label': paf['condition'].astype(str).str.lower().to_numpy(),
                            'ent_s': pd.to_numeric(paf['entS'], errors='coerce').to_numpy(float),
                            'onset': pd.to_numeric(day['onsets_ms'], errors='coerce').to_numpy(float)})
        # onsets_ms holds seconds in some tables and milliseconds in others; the gap between neighbouring
        # syllables of a song tells which (a fraction of a second either way)
        gaps = one.sort_values(['song_name', 'onset']).groupby('song_name', sort=False)['onset'].diff()
        one['onset_s'] = one['onset'] * (1e-3 if gaps.median() > 5 else 1.0)
        frames.append(one)
if misaligned:
    raise RuntimeError('the two BioSound files of these birds do not hold the same rows in the same order, so entS '
                       'cannot be given a song name: ' + ', '.join(misaligned)
                       + '. entS then has to be read from the full BioSound tables.')
bio = pd.concat(frames, ignore_index=True)
del frames
scored['song_name'] = np.where(scored['species'] == 'zebra_finch',
                               zf_song_name(scored['level_bird'], scored['level_day'], scored['level_bout']),
                               scored['wav_file'].map(song_stem))
bio_rows = bio.groupby('song_name', sort=False).indices          # song -> its rows in bio
bio_onset, bio_ent, bio_label = bio['onset_s'].to_numpy(), bio['ent_s'].to_numpy(), bio['label'].to_numpy()

# ---- match the syllables of every song. The amplitude file (Cell L1) holds the onset of every syllable of
# every song, from the syllable tables.
amp = load_pickle(LEVEL_PATH)['syllables']
amp = amp.astype({'species': str, 'bird': str, 'day': str, 'song': str})
amp['song_name'] = np.where(amp['species'] == 'zebra_finch',
                            zf_song_name(amp['bird'], amp['day'], amp['song'].str.extract(r'_bout(.+)$', expand=False)),
                            amp['song'].map(song_stem))
amp_onsets = {name: np.sort(v.to_numpy(float)) for name, v in amp.groupby('song_name', sort=False)['onset_s']}

per_song = []
for name, condition in zip(scored['song_name'], scored['condition']):
    rows, a = bio_rows.get(name), amp_onsets.get(name, np.array([]))
    one = dict(n_table=len(a), n_biosound=0, n_both=0, ent_s=np.nan, label_ok=False)
    if rows is not None:
        found = match_onsets(a, bio_onset[rows])
        both = rows[found[found >= 0]]
        values = bio_ent[both][np.isfinite(bio_ent[both])]
        labels = pd.Series(bio_label[rows])
        one.update(n_biosound=len(rows), n_both=len(both), label_ok=labels.mode().iat[0] == condition,
                   ent_s=float(values.mean()) if len(values) else np.nan)
    per_song.append(one)
scored = pd.concat([scored, pd.DataFrame(per_song)], axis=1)
smaller = scored[['n_table', 'n_biosound']].min(axis=1)
scored['identical'] = (scored['n_both'] == scored['n_table']) & (scored['n_both'] == scored['n_biosound']) & (scored['n_both'] > 0)
# verified: at least half of the syllables are the same ones (and at least three, in songs that have three)
scored['verified'] = (scored['n_both'] >= MIN_SHARED * smaller) & (scored['n_both'] >= smaller.clip(upper=3)) & (smaller > 0)
scored['status'] = np.where(scored['n_biosound'] == 0, 'not in BioSound',
                   np.where(~scored['label_ok'], 'condition differs',
                   np.where(scored['ent_s'].isna(), 'no syllable in both tables', 'ok')))
scored.loc[scored['status'] != 'ok', 'ent_s'] = np.nan

report = scored.groupby(['species', 'bio_bird'], sort=False).agg(
    songs=('status', 'size'), in_biosound=('n_biosound', lambda n: int((n > 0).sum())),
    same_condition=('label_ok', 'sum'), identical=('identical', 'sum'), verified=('verified', 'sum'),
    syllables=('n_table', 'sum'), with_entS=('n_both', 'sum'), biosound_syllables=('n_biosound', 'sum')).astype(int)
report['biosound_only'] = report['biosound_syllables'] - report['with_entS']
report['pct_with_entS'] = (100 * report['with_entS'] / report['syllables'].clip(lower=1)).round(1)
print('songs of Fig. 6I in BioSound. identical: the same syllables in both tables. verified: at least half the same.')
print('syllables: in the syllable table. with_entS: also in BioSound. biosound_only: in BioSound and not in the table.')
print(report.drop(columns='biosound_syllables').to_string())

# ---- stop if anything is off; song_covariates is left as it was ------------------------------------------
found = report['in_biosound'] / report['songs']
verified = report['verified'] / report['in_biosound'].clip(lower=1)
trouble = []
for (species, bird), share in found.items():
    if share < 0.5:
        # the two names side by side, so the difference in how they are written can be seen
        here = scored.loc[(scored['bio_bird'] == bird) & (scored['n_biosound'] == 0), 'song_name'].iloc[0]
        there = bio.loc[bio['bio_bird'] == bird, 'song_name'].iloc[0]
        print(f'  [CHECK] {bird}: a song is named   {here}   here\n'
              f'          {"":{len(bird)}}  and           {there}   in its BioSound file')
        trouble.append(f'{bird}: song names do not match')
    elif verified[(species, bird)] < MIN_VERIFIED:
        # one song that did not verify, as onset-to-onset intervals in ms, to show how the two tables differ
        r = scored[(scored['bio_bird'] == bird) & ~scored['verified'] & (scored['n_biosound'] > 0)].iloc[0]
        gaps_table = np.round(1000 * np.diff(amp_onsets.get(r['song_name'], np.array([]))))[:12].astype(int).tolist()
        gaps_bio = np.round(1000 * np.diff(np.sort(bio_onset[bio_rows[r['song_name']]])))[:12].astype(int).tolist()
        print(f'  [CHECK] {bird}, song {r["song_name"]}: {r["n_table"]} syllables in the table, {r["n_biosound"]} in '
              f'BioSound, {r["n_both"]} the same\n          table intervals (ms)    {gaps_table}\n'
              f'          BioSound intervals (ms) {gaps_bio}')
        trouble.append(f'{bird}: only {100 * verified[(species, bird)]:.0f}% of its songs share their syllables with BioSound')
missing = scored[scored['status'] != 'ok']
if len(missing) and not trouble:
    print(f'\n{len(missing)} songs have no entS:')
    for r in missing.itertuples():
        print(f'   {r.status:22s} {r.bio_bird:12s} {r.condition:7s} {r.wav_file}')
    trouble.append(f'{len(missing)} songs of Fig. 6I have no entS (listed above). All three covariates have to sit on '
                   'the same songs, so these would have to leave the panel, and S8 C-D with them')
if trouble:
    raise RuntimeError('entS was not added. ' + '; '.join(trouble) + '. The lines printed above show where.')

# ---- add the column -----------------------------------------------------------------------------------------
song_covariates = song_covariates.assign(mean_ent_s=scored['ent_s'].to_numpy())
ENT_S_INFO = {'files': 'biosound_paf_slim/<bird>_paf.pickle (entS) and biosound_slim_day/<bird>_slim.pickle (song names), '
                       'written by data_figs Cells A4 and E3',
              'syllables': 'those in both BioSound and the syllable tables, matched on onset within each song',
              'songs': int(len(scored)), 'songs_with_the_same_syllables_in_both_tables': int(scored['identical'].sum()),
              'syllables_in_the_songs': int(scored['n_table'].sum()), 'syllables_with_entS': int(scored['n_both'].sum())}

by_bird = song_covariates.groupby(['species', 'bird_uid', 'condition'])['mean_ent_s'].mean().unstack('condition')
print(f'\nmean_ent_s added to {len(song_covariates)} songs; Regair minus Heliox, mean and SEM across birds:')
print((by_bird['regair'] - by_bird['heliox']).groupby('species').agg(['mean', 'sem', 'size']).round(4).to_string())
# How alike the two entropies are: correlation across songs within each bird. They are different measures,
# so this is for information and nothing depends on it.
r_within = song_covariates.groupby(['species', 'bird_uid']).apply(
    lambda g: g['mean_ent_s'].corr(g['mean_spectral_entropy']))
print('\ncorrelation with mean_spectral_entropy across songs, bird by bird:')
print(r_within.groupby('species').agg(['min', 'median', 'max']).round(2).to_string())
require_complete(song_covariates, 'song_covariates', 'bird_uid', 'condition', species_col='species')

songs of Fig. 6I in BioSound. identical: the same syllables in both tables. verified: at least half the same.
syllables: in the syllable table. with_entS: also in BioSound. biosound_only: in BioSound and not in the table.
                         songs  in_biosound  same_condition  identical  verified  syllables  with_entS  biosound_only  pct_with_entS
species     bio_bird                                                                                                                
starling    s_b1349h_23    317          317             317        317       317       8450       8450              0          100.0
            s_b1354_25     142          142             142        142       142       4280       4280              0          100.0
            s_b1364h_25    320          320             320        319       320      15776      15775              0          100.0
            s_b1481h_24     49           49              49         48        49        946        945           

## 7. Assemble and save

Tables are trimmed to the columns the figures use and checked for anything that would
tie the file to this machine (path objects, absolute paths).

In [18]:
# Cell 13: trim, check, save
def as_plain(df, columns, categorical=()):
    """Keep `columns`, reset the index, turn categoricals into plain strings. Long string
    columns named in `categorical` are stored as categories to keep the file small."""
    out = df[list(columns)].reset_index(drop=True).copy()
    for c in out.columns:
        if str(out[c].dtype) == 'category':
            out[c] = out[c].astype(str)
    for c in categorical:
        out[c] = out[c].astype('category')
    return out


def assert_portable(name, df):
    """No path objects and no absolute paths: either would tie the pickle to this cluster."""
    for c in df.columns:
        if not (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])):
            continue
        vals = pd.Series(df[c].dropna().unique())
        if vals.map(lambda v: isinstance(v, os.PathLike)).any():
            raise TypeError(f'{name}.{c} holds path objects; store the file name as text')
        if vals.map(lambda v: isinstance(v, str) and os.sep in v).any():
            raise ValueError(f'{name}.{c} holds file paths; store the file name only')


# bout_index is a file path in the starling tables, so it is replaced by an integer code.
# The k-gram step only needs to know which syllables share a bout. Missing stays missing,
# so those rows are left out of bouts exactly as before.
bout_code = pd.factorize(starling_syllable_nll['bout'])[0].astype(float)
bout_code[bout_code < 0] = np.nan
sn_out = starling_syllable_nll.assign(bout=bout_code)

# The amplitude columns come from Cell L1. Saving without them would leave Fig. 6I and S8 C-D with
# nothing to adjust for, so that is an error here and not a missing column in the figure notebook.
if 'mean_level_db' not in song_covariates.columns or 'level_db' not in song_trace_metrics.columns:
    raise RuntimeError('song amplitude is missing: run Cell L1 before this cell')
# The same goes for the BioSound entropy of Fig. 6I, which comes from Cell B1.
if 'mean_ent_s' not in song_covariates.columns:
    raise RuntimeError('BioSound spectral entropy is missing: run Cell B1 before this cell')

# bird_uid is kept for Fig. 6D whenever the exports carry it (see Cell 4)
pos_cols = [c for c in ['species', 'bird_uid', 'condition', 'bin', 'nll'] if c in syllable_position_nll.columns]

bundle = {
    'region_nll': as_plain(region_nll, ['species', 'bird_uid', 'condition', 'region', 'nll']),
    'boundary_nll': as_plain(boundary_nll, ['species', 'bird_uid', 'condition', 'region', 'nll'],
                             categorical=['species', 'bird_uid', 'condition', 'region']),
    'syllable_position_nll': as_plain(syllable_position_nll, pos_cols,
                                      categorical=[c for c in pos_cols if c not in ('bin', 'nll')]),
    # The old mean_power, power and amplitude columns are not saved. None of them measured how loud a song
    # was, no figure uses them, and a column called amplitude that is not the amplitude in the paper would
    # invite a mix-up. They are still in the export pickles and in s4_figure_cache.
    'song_covariates': as_plain(song_covariates, ['species', 'bird_uid', 'condition', 's4_avg_nll',
                                                  'mean_level_db', 'mean_ent_s', 'duration_s',
                                                  'mean_spectral_entropy']),
    'starling_syllable_nll': as_plain(sn_out, ['bird', 'cond2', 'basename', 'bout', 'pos', 'label',
                                               'nll_syl', 'nll_gap']),
    'starling_kgram_sets': starling_kgram_sets,
    'zf_syllable_nll': as_plain(zf_syllable_nll, ['bird_id', 'condition', 'song_id', 'label',
                                                  'is_intro', 'interior_nll', 'gap_nll']),
    'song_trace_metrics': as_plain(song_trace_metrics, ['species', 'bird', 'bird_uid', 'condition',
                                                        'wav_file', 'nll_p100', 'jump_rate_raw', 'level_db']),
}
for name, obj in bundle.items():
    if isinstance(obj, pd.DataFrame):
        assert_portable(name, obj)

bundle['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'built_by': 's4_nll_build_data.ipynb',
    'audio_tables': 'recomputed from audio' if FROM_AUDIO else 'reused from s4_figure_cache',
    'params': PARAMS,
    'kgram_selection': {'prevalence': motif_export.get('THRESHOLD'), 'fold': motif_export.get('FOLD_CUT_FINAL')},
    'kgram_birds': kgram_birds,
    'amplitude': AMPLITUDE_INFO,
    'ent_s': ENT_S_INFO,
    'incomplete_tables': list(INCOMPLETE),
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
    'units': {'nll': 'nats per audio sample (S4 negative log-likelihood)',
              'starling_syllable_nll': 'mean of 5 ms bins over the syllable (nll_syl) or the following gap (nll_gap)',
              'zf_syllable_nll': 'mean over the syllable minus 10 ms at each end (interior_nll) or the following gap (gap_nll)',
              'song_trace_metrics': 'nll_p100 = peak over song bins; jump_rate_raw = fraction of bin-to-bin jumps; '
                                    'level_db = amplitude of the song: the mean over its syllables of the RMS '
                                    'amplitude of the recorded waveform, dB re full scale',
              'song_covariates': 'mean_level_db = amplitude of the song, the same measure as level_db in '
                                 'song_trace_metrics; mean_ent_s = mean over the syllables of the song of entS, '
                                 'the spectral entropy BioSound computes (the entropy in Fig. 6I); '
                                 'duration_s = length of the song in seconds; mean_spectral_entropy = mean over '
                                 'the syllables of the spectral_entropy column of the syllable tables (the '
                                 'entropy Fig. 6I used before, kept for comparison)'},
}

os.makedirs(os.path.dirname(BUNDLE_PATH) or '.', exist_ok=True)
with open(BUNDLE_PATH, 'wb') as f:
    pickle.dump(bundle, f, protocol=4)      # protocol 4 opens on every Python 3 still in use

print(f'saved {BUNDLE_PATH}  ({os.path.getsize(BUNDLE_PATH) / 1e6:.1f} MB)\n')
bird_cols = {'region_nll': 'bird_uid', 'boundary_nll': 'bird_uid', 'syllable_position_nll': 'bird_uid',
             'song_covariates': 'bird_uid', 'starling_syllable_nll': 'bird', 'zf_syllable_nll': 'bird_id',
             'song_trace_metrics': 'bird_uid'}
for name, obj in bundle.items():
    if isinstance(obj, pd.DataFrame):
        n_birds = obj[bird_cols[name]].nunique() if bird_cols.get(name) in obj.columns else '-'
        print(f'  {name:24s} {len(obj):>8d} rows  {obj.shape[1]:>2d} columns  {n_birds} birds')
print(f'  {"starling_kgram_sets":24s} {len(starling_kgram_sets):>8d} sets')
if INCOMPLETE:
    print('\n[INCOMPLETE] saved with tables short of birds:')
    for message in INCOMPLETE:
        print('   ', message)

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/s4_figure_data.pkl  (26.6 MB)

  region_nll                   4188 rows   5 columns  12 birds
  boundary_nll               137922 rows   5 columns  12 birds
  syllable_position_nll     1380720 rows   5 columns  12 birds
  song_covariates              2094 rows   8 columns  12 birds
  starling_syllable_nll       28057 rows   8 columns  6 birds
  zf_syllable_nll             34051 rows   7 columns  6 birds
  song_trace_metrics           2094 rows   8 columns  12 birds
  starling_kgram_sets            60 sets
